# Common Libraries

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from mujoco_util import enforce_equality_constraints
from vis_3D import Plotter3D, make_mesh_ds
from mujoco_util import get_mesh

# EasyMocap SMPL loader
EASYMOCAP_DIR = "/home/seojin/Tools/pose_estimation/EasyMocap"
if os.path.isdir(EASYMOCAP_DIR) and EASYMOCAP_DIR not in sys.path:
    sys.path.insert(0, EASYMOCAP_DIR)
from easymocap.smplmodel import load_model

# Params

In [3]:
# Mujoco
model_path = "/mnt/sdb2/DeepProprioception/Projects/DP02_mri/Myosuite/muscle_mimic_mocap/auto_marker/Scale/aligned3.mjb"
translation_path = "/mnt/sdb2/DeepProprioception/Projects/DP02_mri/Myosuite/muscle_mimic_mocap/auto_marker/Scale/aligned3_info.json"

# SMPL
body_model_type = "smplx"
template_mesh_path = "/mnt/sdb2/DeepProprioception/Projects/DP02_emg/Myosuite/muscle_mimic_mocap/auto_marker/Scale/template_matching_skin_alignedMM.npz"
smpl_model_dir = "/home/seojin/Tools/pose_estimation/EasyMocap/data/smplx"

# Experiment
subj_number = "2"
participant_info_path = "/mnt/sdb2/DeepProprioception/Exp_info/participant_info.csv"

# Coords
ref_coords = "RAI"
visualize_coord_order = "LAS"

# Constants

In [4]:
# SMPL
smpl_joints = [ "Pelvis", 
               "L_Hip", "R_Hip", 
               "Spine1", 
               "L_Knee", "R_Knee",
               "Spine2", 
               "L_Ankle", "R_Ankle", 
               "Spine3", 
               "L_Foot", "R_Foot", 
               "Neck", 
               "L_Collar", "R_Collar", 
               "Head", 
               "L_Shoulder", "R_Shoulder", 
               "L_Elbow", "R_Elbow", 
               "L_Wrist", "R_Wrist", 
               "L_Hand", "R_Hand" ]

# SMPL-X
smplx_joints = [ "Pelvis",
                    "L_Hip", "R_Hip",
                    "Spine1", 
                    "L_Knee", "R_Knee",
                    "Spine2",
                    "L_Ankle", "R_Ankle",
                    "Spine3",
                    "L_Foot", "R_Foot",
                    "Neck",
                    "L_Collar", "R_Collar",
                    "Head",
                    "L_Shoulder", "R_Shoulder",
                    "L_Elbow", "R_Elbow",
                    "L_Wrist", "R_Wrist",
                    "Jaw",
                    "L_Eye", "R_Eye",
                    "L_Index1", "L_Index2", "L_Index3",
                    "L_Middle1", "L_Middle2", "L_Middle3",
                    "L_Pinky1", "L_Pinky2", "L_Pinky3",
                    "L_Ring1", "L_Ring2", "L_Ring3",
                    "L_Thumb1", "L_Thumb2", "L_Thumb3",
                    "R_Index1", "R_Index2", "R_Index3",
                    "R_Middle1", "R_Middle2", "R_Middle3",
                    "R_Pinky1", "R_Pinky2", "R_Pinky3",
                    "R_Ring1", "R_Ring2", "R_Ring3",
                    "R_Thumb1", "R_Thumb2", "R_Thumb3" ]

In [5]:
DEFAULT_BONE_TO_SMPL_JOINTS = {
    # Thorax covers core body, head, lower body, and left upper limb
    "thorax": ['Pelvis', 'L_Hip', 'R_Hip', 'Spine1', 'L_Knee', 'R_Knee', 'Spine2',
               'L_Ankle', 'R_Ankle', 'Spine3', 'L_Foot', 'R_Foot', 'Neck',
               'L_Collar', 'Head', 'L_Shoulder', 'L_Elbow', 'L_Wrist', 'L_Hand'],
    
    # Right collar / clavicle
    "clavicle_r": ["R_Collar"],
    
    # Right upper arm (Humerus)
    "humerus_r": ["R_Shoulder"],
    
    # Right forearm: split elbow weight
    "ulna_r": ["R_Elbow"],
    
    # Right hand / wrist
    "lunate_r" : ["R_Wrist", "R_Hand"]
}

DEFAULT_BONE_TO_SMPLX_JOINTS = {
    # Thorax covers core body, head, face (jaw/eyes), lower body, left arm, and left hand (25..39)
    "thorax": ['Pelvis', 'L_Hip', 'R_Hip', 'Spine1', 'L_Knee', 'R_Knee', 'Spine2',
               'L_Ankle', 'R_Ankle', 'Spine3', 'L_Foot', 'R_Foot', 'Neck',
               'L_Collar', 'Head', 'L_Shoulder',
               'L_Elbow', 'L_Wrist', 'Jaw', 'L_Eye',
               'R_Eye', 'L_Index1', 'L_Index2', 'L_Index3', 'L_Middle1',
               'L_Middle2', 'L_Middle3', 'L_Pinky1', 'L_Pinky2', 'L_Pinky3',
               'L_Ring1', 'L_Ring2', 'L_Ring3', 'L_Thumb1', 'L_Thumb2',
               'L_Thumb3'],
    
    # Right collar / clavicle
    "clavicle_r": ["R_Collar"],
    
    # Right upper arm (Humerus)
    "humerus_r": ["R_Shoulder"],
    
    # Right forearm: split elbow weight
    "ulna_r": ["R_Elbow"],
    
    # Right hand / wrist
    "lunate_r" : ["R_Wrist"],
    
    # Right Thumb
    "firstmc_r": ["R_Thumb1"],
    "proximal_thumb_r": ["R_Thumb2"],
    "distal_thumb_r": ["R_Thumb3"],

    # Right Index
    "2proxph_r": ["R_Index1"],
    "2midph_r": ["R_Index2"],
    "2distph_r": ["R_Index3"],

    # Right Middle
    "3proxph_r": ["R_Middle1"],
    "3midph_r": ["R_Middle2"],
    "3distph_r": ["R_Middle3"],

    # Right Ring
    "4proxph_r": ["R_Ring1"],
    "4midph_r": ["R_Ring2"],
    "4distph_r": ["R_Ring3"],

    # Right Pinky
    "5proxph_r": ["R_Pinky1"],
    "5midph_r": ["R_Pinky2"],
    "5distph_r": ["R_Pinky3"],
}

# Load

## Gender

In [6]:
participant_info = pd.read_csv(participant_info_path)
gender = participant_info[participant_info["Nick name"] == f"DP{subj_number.zfill(2)}"]["Gender"].iloc[0].lower()
print(f"Subject: DP{subj_number.zfill(2)}, Gender: {gender}")

Subject: DP02, Gender: female


## Model translation

In [7]:
translation_info = json.load(open(translation_path, "r"))
translation = np.array(translation_info["model_delta_pos"])

## SMPL

In [8]:
# Template vertices
template_mesh_info = np.load(template_mesh_path, allow_pickle = True)
template_mesh_info = template_mesh_info[body_model_type].item()
kp_labels = template_mesh_info["keypoint_names"]
smpl_verts = template_mesh_info["vertices"] + translation
n_vertices = len(smpl_verts)

# Template face
smpl_faces = template_mesh_info["faces"]

# Template Keypoint
smpl_kp = template_mesh_info["keypoints"]

# Homogenouse
V_0_homo = np.hstack([smpl_verts, np.ones((n_vertices, 1), dtype=smpl_verts.dtype)])


# Coord
smpl_coords = template_mesh_info["coords"]

# Qpos
reference_qpos = template_mesh_info["qpos"]

In [9]:
# SMPL model
body_model = load_model(gender = gender if "gender" in locals() else "female", 
                        model_type = body_model_type, 
                        model_path = smpl_model_dir)
smpl_weights = body_model.weights.cpu().numpy() if hasattr(body_model.weights, "cpu") else np.array(body_model.weights)

## MJC model

In [10]:
# Load model
model = mujoco.MjModel.from_binary_path(model_path)

# Set default pose
mj_data = mujoco.MjData(model)
mj_data.qpos[:] = reference_qpos
enforce_equality_constraints(model, mj_data)
mujoco.mj_forward(model, mj_data)

# Mesh
mesh_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_MESH, i) for i in range(model.nmesh)]

# Skeleton
skeleton_names = [name for name in mesh_names if name not in ["mri_mesh", "arm_marker_mesh"]]

# Preprocess

## Mapping

In [11]:
if body_model_type == "smplx":
    bone_to_smpl_joints = DEFAULT_BONE_TO_SMPLX_JOINTS
    smpl_joints = smplx_joints
else:
    bone_to_smpl_joints = DEFAULT_BONE_TO_SMPL_JOINTS
    smpl_joints = smpl_joints

In [12]:
bone_names = list(bone_to_smpl_joints.keys())
bone_body_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, b) for b in bone_names]
n_bones = len(bone_names)

## Build W_mujoco: (N, num_bones)

In [13]:
bone_to_vertex_weights = np.zeros((n_vertices, n_bones), dtype=np.float64)
for bone_i, bone_name in enumerate(bone_names):
    joint_specs = bone_to_smpl_joints[bone_name]
    for spec in joint_specs:
        joint_i, scale = smpl_joints.index(spec), 1.0
        
        bone_to_vertex_weights[:, bone_i] += smpl_weights[:, joint_i] * scale

    # Row-wise normalize to ensure weights sum to 1.0 for each vertex
    row_sums = bone_to_vertex_weights.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    bone_to_vertex_weights /= row_sums

In [14]:
T_0_inv = np.zeros((n_bones, 4, 4), dtype=np.float64)
for i, bone_i in enumerate(bone_body_ids):
    T_0 = np.eye(4, dtype=np.float64)
    T_0[:3, :3] = mj_data.xmat[bone_i].reshape(3, 3)
    T_0[:3, 3] = mj_data.xpos[bone_i]
    T_0_inv[i] = np.linalg.inv(T_0)

## Deform

In [15]:
# 1. Update MuJoCo kinematics
qpos = np.zeros(len(mj_data.qpos))
mj_data.qpos[:] = qpos
enforce_equality_constraints(model, mj_data)
mujoco.mj_forward(model, mj_data)

In [16]:
# 2. Compute delta_T for each bone
delta_T = np.zeros((n_bones, 4, 4), dtype=np.float64)
for i, bone_i in enumerate(bone_body_ids):
    T_new = np.eye(4, dtype=np.float64)
    T_new[:3, :3] = mj_data.xmat[bone_i].reshape(3, 3)
    T_new[:3, 3] = mj_data.xpos[bone_i]
    delta_T[i] = T_new @ T_0_inv[i]

In [17]:
# 3. Linear Blend Skinning (vectorized via np.einsum)
vertex_transforms = np.einsum('vm,mij->vij', bone_to_vertex_weights, delta_T)
V_new = np.einsum('vij,vj->vi', vertex_transforms, V_0_homo)[:, :3]

# Visualization

In [25]:
skeleton_ds_list = []
for name in skeleton_names:
    verts, faces = get_mesh(model, mj_data, name)
    
    for v in verts:
        skeleton_ds_list.append(make_mesh_ds(vertices=v,
                                             faces=faces,
                                             name=name,
                                             color="#FFFFF0",
                                             opacity=0.95,
                                             coord_order = ref_coords))

In [26]:
mesh_ds = make_mesh_ds(vertices=V_new,
                       faces=smpl_faces,
                       coord_order=ref_coords,
                       times=[0],
                       name="SMPL_Skin",
                       color="#e0ac69",
                       opacity=0.3)

In [27]:
plotter = Plotter3D(visualize_coord_order = visualize_coord_order)

In [28]:
plotter.plot_single_dataset(mesh_ds_list = [mesh_ds] + mesh_ds_list)